# Configuration summary tables

Builds the three derived tables behind the data management dashboard:

```
primary ∪ secondary ──► by_location ──► (rollup) ──► configurations_summary
                             │
                             └──► join_geometry ──► configurations_by_location

location_attributes ──► (pivot) ──► ⋈ locations ──► locations_with_attributes
```

Both tables declare their own filterable dimensions (`group_by`) and value columns
(`metrics`) as Iceberg table properties, which is how the OGC API serves them
through the generic `/collections/{id}/items` route without any per-table code.

In [ ]:
import logging

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.querying.utils import join_geometry

logger = logging.getLogger()

In [ ]:
%%time
spark = create_spark_session(
    aws_profile="admin-user",
    start_spark_cluster=True,
    executor_instances=48,
    executor_memory="32g",
    executor_cores=4
)
ev = teehr.RemoteReadWriteEvaluation(
    spark=spark,
    enable_spark_proxy=True
)

In [ ]:
BY_LOCATION_TABLE_NAME = "configurations_by_location"
SUMMARY_TABLE_NAME = "configurations_summary"

# group_by / metrics become Iceberg table properties. The API reads them to
# decide which columns are filterable dimensions and which are values, so
# these lists are the contract between the warehouse and the OGC endpoints.
# 'name' and 'geometry' belong to neither list by convention.
BY_LOCATION_GROUP_BY = [
    "primary_location_id",
    "configuration_name",
    "variable_name",
    "unit_name",
]
BY_LOCATION_METRICS = [
    "min_reference_time",
    "max_reference_time",
    "min_value_time",
    "max_value_time",
    "num_members",
]
BY_LOCATION_DESCRIPTION = (
    "Configurations, variables and units available at each primary location, "
    "with timeseries time ranges and ensemble member counts"
)

SUMMARY_GROUP_BY = [
    "configuration_name",
    "variable_name",
    "unit_name",
    "timeseries_type",
]
SUMMARY_METRICS = [
    "min_reference_time",
    "max_reference_time",
    "min_value_time",
    "max_value_time",
    "num_locations",
    "description",
]
SUMMARY_DESCRIPTION = (
    "Per-configuration summary of available timeseries: variables, units, "
    "time ranges and location counts"
)

LOCATIONS_TABLE_NAME = "locations_with_attributes"

# The attribute columns are discovered from the data rather than listed here,
# and go in 'metrics': they are values of a location, not dimensions. Keeping
# them out of 'group_by' also holds the API's ORDER BY to two columns while
# still listing every attribute in /queryables for column discovery.
LOCATIONS_GROUP_BY = [
    "location_id",
    "name",
]
LOCATIONS_DESCRIPTION = (
    "One row per location with its attributes pivoted into columns"
)

In [ ]:
def summarize_primary_locations(
    ev: teehr.Evaluation
) -> None:
    """Summarize primary locations."""
    logger.info(
        "Summarizing primary locations into a spark dataframe..."
    )
    return ev.spark.sql("""
        SELECT
            location_id as primary_location_id,
            configuration_name,
            variable_name,
            unit_name,
            MIN(reference_time) AS min_reference_time,
            MAX(reference_time) AS max_reference_time,
            MIN(value_time)     AS min_value_time,
            MAX(value_time)     AS max_value_time,
            CAST(NULL AS BIGINT) AS num_members
        FROM iceberg.teehr.primary_timeseries
        GROUP BY primary_location_id, configuration_name, variable_name, unit_name
    """)


def summarize_secondary_locations(
    ev: teehr.Evaluation
) -> None:
    """Summarize secondary locations."""
    logger.info(
        "Summarizing secondary locations into a spark dataframe..."
    )
    return ev.spark.sql("""
        SELECT
            cf.primary_location_id,
            st.configuration_name,
            st.variable_name,
            st.unit_name,
            MIN(st.reference_time) AS min_reference_time,
            MAX(st.reference_time) AS max_reference_time,
            MIN(st.value_time)     AS min_value_time,
            MAX(st.value_time)     AS max_value_time,
            COUNT(DISTINCT st.member) AS num_members
        FROM iceberg.teehr.secondary_timeseries st
        JOIN iceberg.teehr.location_crosswalks cf
            ON cf.secondary_location_id = st.location_id
        GROUP BY cf.primary_location_id, st.configuration_name, st.variable_name, st.unit_name
    """)

In [ ]:
def summarize_configurations(
    ev: teehr.Evaluation,
    by_location_sdf
) -> None:
    """Roll the per-location summary up to one row per configuration.

    Derived from the pre-geometry by-location frame so that num_locations counts
    every location with timeseries, not only those carrying geometry.
    """
    logger.info(
        "Rolling the location summary up to one row per configuration..."
    )
    by_location_sdf.createOrReplaceTempView("by_location")
    return ev.spark.sql("""
        WITH agg AS (
            SELECT
                configuration_name,
                variable_name,
                unit_name,
                MIN(min_reference_time) AS min_reference_time,
                MAX(max_reference_time) AS max_reference_time,
                MIN(min_value_time)     AS min_value_time,
                MAX(max_value_time)     AS max_value_time,
                COUNT(DISTINCT primary_location_id) AS num_locations
            FROM by_location
            GROUP BY configuration_name, variable_name, unit_name
        )
        SELECT
            agg.*,
            c.description,
            c.timeseries_type
        FROM agg
        JOIN iceberg.teehr.configurations c
            ON c.name = agg.configuration_name
    """)


def summarize_locations_with_attributes(
    ev: teehr.Evaluation
) -> None:
    """Pivot location attributes into one row per location.

    location_attributes_view() does the long-to-wide pivot; the join adds
    'name' and is a LEFT join so locations without attributes still appear.
    """
    logger.info(
        "Pivoting location attributes into a spark dataframe..."
    )
    attributes_sdf = ev.location_attributes_view().to_sdf()
    locations_sdf = ev.locations.to_sdf().selectExpr("id AS location_id", "name")
    return locations_sdf.join(attributes_sdf, on="location_id", how="left")

In [ ]:
by_location_sdf = summarize_primary_locations(ev=ev).unionByName(
    summarize_secondary_locations(ev=ev)
)

configurations_summary_sdf = summarize_configurations(
    ev=ev,
    by_location_sdf=by_location_sdf
)

# join_geometry adds 'name' and 'geometry' from the locations table. It joins
# inner, and locations.geometry is nullable, so rows without a mappable location
# drop out here -- they could not be drawn or clicked through on the map anyway.
by_location_with_geometry_sdf = join_geometry(
    by_location_sdf, ev.locations.to_sdf()
).filter("geometry IS NOT NULL")

locations_with_attributes_sdf = summarize_locations_with_attributes(ev=ev)

In [ ]:
%%time
# create_or_replace (not overwrite) because the schema changes between runs.
# Partitioned on configuration_name: the dashboard's hot query filters on it,
# and that is the query carrying geometry for every row.
ev._write.to_warehouse(
    source_data=by_location_with_geometry_sdf,
    table_name=BY_LOCATION_TABLE_NAME,
    write_mode="create_or_replace",
    partition_by=["configuration_name"],
    write_ordered_by=BY_LOCATION_GROUP_BY,
)

In [ ]:
%%time
ev._write.to_warehouse(
    source_data=configurations_summary_sdf,
    table_name=SUMMARY_TABLE_NAME,
    write_mode="create_or_replace",
    write_ordered_by=SUMMARY_GROUP_BY,
)

# create_or_replace because the attribute set -- and therefore the schema --
# changes as attributes are added to the warehouse.
ev._write.to_warehouse(
    source_data=locations_with_attributes_sdf,
    table_name=LOCATIONS_TABLE_NAME,
    write_mode="create_or_replace",
    write_ordered_by=["location_id"],
)

In [ ]:
def set_table_properties(
    ev: teehr.Evaluation,
    table_name: str,
    properties: dict
) -> None:
    """Set Iceberg table properties for a given table in the warehouse."""
    logger.info(f"Setting table properties for {table_name}...")
    for key, value in properties.items():
        ev.spark.sql(f"""
            ALTER TABLE iceberg.teehr.{table_name} SET TBLPROPERTIES ('{key}' = '{value}')
        """)


# Must run after the writes: create_or_replace drops the table, taking any
# previously set properties with it.
set_table_properties(
    ev=ev,
    table_name=BY_LOCATION_TABLE_NAME,
    properties={
        "description": BY_LOCATION_DESCRIPTION,
        "group_by": ", ".join(BY_LOCATION_GROUP_BY),
        "metrics": ", ".join(BY_LOCATION_METRICS),
    },
)
set_table_properties(
    ev=ev,
    table_name=SUMMARY_TABLE_NAME,
    properties={
        "description": SUMMARY_DESCRIPTION,
        "group_by": ", ".join(SUMMARY_GROUP_BY),
        "metrics": ", ".join(SUMMARY_METRICS),
    },
)

# Attribute columns are whatever the pivot produced.
locations_metrics = [
    c for c in locations_with_attributes_sdf.columns
    if c not in LOCATIONS_GROUP_BY
]
locations_properties = {
    "description": LOCATIONS_DESCRIPTION,
    "group_by": ", ".join(LOCATIONS_GROUP_BY),
}
if locations_metrics:
    locations_properties["metrics"] = ", ".join(locations_metrics)
set_table_properties(
    ev=ev,
    table_name=LOCATIONS_TABLE_NAME,
    properties=locations_properties,
)